# Procedural Info Theory — Analysis

Load computed certificates (edits, modules, motifs), summarize distributions, and run diversity analysis when distance matrices are available.

**Inputs:** `output/datasets/swe_bench_lite/test.parquet` or `output/hf_export/swe_bench_lite/test.json`  
**Plots:** `notebooks/plots/`

## Setup

In [ ]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path("../").resolve()
DATA_DIR = ROOT / "output"
PLOTS_DIR = Path("plots").resolve()
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Prefer parquet; fallback to JSON
parquet_path = DATA_DIR / "datasets" / "swe_bench_lite" / "test.parquet"
json_path = DATA_DIR / "hf_export" / "swe_bench_lite" / "test.json"

## Load Data

In [ ]:
def load_records() -> pd.DataFrame:
    if parquet_path.exists():
        df = pd.read_parquet(parquet_path)
        for col in ["edits", "modules", "motifs"]:
            if col in df.columns and df[col].dtype == object:
                df[col] = df[col].apply(lambda x: json.loads(x) if isinstance(x, str) else x)
        return df
    if json_path.exists():
        with open(json_path) as f:
            data = json.load(f)
        return pd.DataFrame(data["records"])
    raise FileNotFoundError(f"No data at {parquet_path} or {json_path}. Run extraction first.")

df = load_records()
df.head()

## Summary Stats

In [ ]:
def edits_delta(edits) -> int:
    if not isinstance(edits, list):
        return 0
    return sum(c.get("delta", 0) for c in edits if isinstance(c, dict))

def edits_ops_count(edits) -> int:
    if not isinstance(edits, list):
        return 0
    return sum(len(c.get("operations", [])) for c in edits if isinstance(c, dict))

def modules_count(modules) -> int:
    return len(modules) if isinstance(modules, list) else 0

def motifs_seq_len(motifs) -> int:
    if isinstance(motifs, dict):
        return len(motifs.get("sequence", []))
    return 0

stats = pd.DataFrame({
    "instance_id": df["instance_id"],
    "repo": df["repo"],
    "edits_delta": df["edits"].apply(edits_delta),
    "edits_ops": df["edits"].apply(edits_ops_count),
    "modules_count": df["modules"].apply(modules_count),
    "motifs_seq_len": df["motifs"].apply(motifs_seq_len),
})
stats.describe()

## Distributions

In [ ]:
import altair as alt

base = alt.Chart(stats).properties(width=200, height=180)

h1 = base.mark_bar().encode(
    alt.X("edits_delta:Q", bin=alt.Bin(maxbins=20), title="delta"),
    alt.Y("count()", title=""),
).properties(title="Edits delta (sum per instance)")

h2 = base.mark_bar().encode(
    alt.X("edits_ops:Q", bin=alt.Bin(maxbins=20), title="ops"),
    alt.Y("count()", title=""),
).properties(title="Edit operations count")

h3 = base.mark_bar().encode(
    alt.X("modules_count:Q", bin=alt.Bin(maxbins=20), title="count"),
    alt.Y("count()", title=""),
).properties(title="Modules (co-edit) tokens")

h4 = base.mark_bar().encode(
    alt.X("motifs_seq_len:Q", bin=alt.Bin(maxbins=20), title="len"),
    alt.Y("count()", title=""),
).properties(title="Motifs sequence length")

chart = alt.vconcat(alt.hconcat(h1, h2), alt.hconcat(h3, h4))
chart.save(PLOTS_DIR / "distributions.png")
chart

## Stratum (Repo) Breakdown

In [ ]:
repo_counts = stats.groupby("repo").agg(
    n=("instance_id", "count"),
    mean_edits_ops=("edits_ops", "mean"),
    mean_modules=("modules_count", "mean"),
    mean_motifs_len=("motifs_seq_len", "mean"),
).reset_index()
repo_counts

In [ ]:
chart = (
    alt.Chart(repo_counts)
    .mark_bar()
    .encode(
        alt.X("repo:N", title="", sort="-y"),
        alt.Y("n:Q", title="instances"),
    )
    .properties(title="Instances per repo (stratum)", width=400, height=250)
)
chart.save(PLOTS_DIR / "stratum_counts.png")
chart

## Null / Empty Rates

In [ ]:
n = len(stats)
empty_edits = (stats["edits_ops"] == 0).sum()
empty_motifs = (stats["motifs_seq_len"] == 0).sum()
empty_modules = (stats["modules_count"] == 0).sum()

pd.DataFrame({
    "representation": ["edits", "modules", "motifs"],
    "empty_count": [empty_edits, empty_modules, empty_motifs],
    "empty_pct": [100 * empty_edits / n, 100 * empty_modules / n, 100 * empty_motifs / n],
})

## Diversity Analysis (when matrices available)

Run `scripts/build_distance_matrices.py` first to produce `distances.parquet` and `labels.parquet`. Then run the cell below.

In [ ]:
import sys
sys.path.insert(0, str(ROOT))

def run_diversity_if_available():
    candidates = [
        DATA_DIR / "distances.parquet",
        DATA_DIR / "matrices.npz",
        DATA_DIR / "datasets" / "swe_bench_lite" / "distances.parquet",
    ]
    matrices_path = next((p for p in candidates if p.exists()), None)
    if not matrices_path:
        print("Skipping: distances.parquet or matrices.npz not found. Run build_distance_matrices.py first.")
        return
    labels_path = matrices_path.parent / "labels.parquet"
    if not labels_path.exists():
        labels_path = matrices_path.parent / "labels.json"
    if not labels_path.exists():
        print("Skipping: labels.parquet or labels.json not found.")
        return

    from analysis.diversity import run_diversity_analysis
    from analysis.io import load_labels, load_matrices

    matrices = load_matrices(matrices_path)
    labels = load_labels(labels_path)

    if len(matrices) < 2 or len(labels) != list(matrices.values())[0].shape[0]:
        print("Invalid matrices or labels.")
        return

    results = run_diversity_analysis(matrices, labels)
    import altair as alt
    import pandas as pd

    names = list(matrices.keys())
    rho_arr = results["rank_correlation"]
    heatmap_data = [
        {"repr_i": names[i], "repr_j": names[j], "rho": float(rho_arr[i][j])}
        for i in range(len(names))
        for j in range(len(names))
    ]

    heatmap = (
        alt.Chart(pd.DataFrame(heatmap_data))
        .mark_rect()
        .encode(
            alt.X("repr_j:N", title=""),
            alt.Y("repr_i:N", title=""),
            alt.Color("rho:Q", scale=alt.Scale(scheme="redblue", domainMid=0), title="ρ"),
        )
        .properties(
            title="Rank correlation (Spearman) between distance vectors",
            width=280,
            height=260,
        )
    )
    heatmap_text = (
        alt.Chart(pd.DataFrame(heatmap_data))
        .mark_text()
        .encode(alt.X("repr_j:N"), alt.Y("repr_i:N"), alt.Text("rho:Q", format=".2f"))
    )
    (heatmap + heatmap_text).save(PLOTS_DIR / "rank_correlation.png")

    sr_df = pd.DataFrame(
        [{"repr": k, "ratio": v if pd.notna(v) else 0} for k, v in results["stratum_ratios"].items()]
    )
    stratum_chart = (
        alt.Chart(sr_df)
        .mark_bar(color="steelblue")
        .encode(alt.X("repr:N", sort="-y", title=""), alt.Y("ratio:Q", title="ratio"))
        .properties(
            title="Within/across stratum ratio (<1 = clusters by stratum)",
            width=280,
            height=220,
        )
    )
    stratum_chart.save(PLOTS_DIR / "stratum_ratios.png")

    sil_df = pd.DataFrame(
        [{"repr": k, "score": v if pd.notna(v) else 0, "metric": "silhouette"} for k, v in results["silhouette_scores"].items()]
    )
    var_df = pd.DataFrame(
        [{"repr": k, "score": v if pd.notna(v) else 0, "metric": "unique_variance"} for k, v in results["unique_variances"].items()]
    )
    sil_chart = (
        alt.Chart(sil_df)
        .mark_bar(color="coral")
        .encode(alt.X("repr:N", sort="-y", title=""), alt.Y("score:Q", title="score"))
        .properties(title="Silhouette (higher = better separation)", width=200, height=220)
    )
    var_chart = (
        alt.Chart(var_df)
        .mark_bar(color="seagreen")
        .encode(alt.X("repr:N", sort="-y", title=""), alt.Y("score:Q", title="variance"))
        .properties(title="Unique variance (higher = less redundant)", width=200, height=220)
    )
    (sil_chart | var_chart).save(PLOTS_DIR / "diversity_scores.png")

run_diversity_if_available()

## Per-Instance Representation Variance

Requires `per_instance_rep_correlation.parquet` (from `run_diversity_analysis.py`). Low mean_rho = instance expressed differently across representations.

In [ ]:
def plot_per_instance_rho():
    pi_path = DATA_DIR / "datasets" / "swe_bench_lite" / "per_instance_rep_correlation.parquet"
    if not pi_path.exists():
        pi_path = DATA_DIR / "per_instance_rep_correlation.parquet"
    if not pi_path.exists():
        print("Skipping: per_instance_rep_correlation.parquet not found.")
        return
    pi_df = pd.read_parquet(pi_path)
    chart = (
        alt.Chart(pi_df.dropna(subset=["mean_rho"]))
        .mark_bar()
        .encode(
            alt.X("mean_rho:Q", bin=alt.Bin(maxbins=25), title="mean Spearman ρ across repr pairs"),
            alt.Y("count()", title="instances"),
        )
        .properties(title="Per-instance representation correlation (low = variable across reprs)", width=350, height=220)
    )
    chart.save(PLOTS_DIR / "per_instance_rho.png")
    chart
plot_per_instance_rho()

## Divergence from Baseline

Requires eval records with tokens + behavioral/mechanistic/functional (from `eval/run_eval.py --save-records`). Cosine distance from inferred embeddings to token baseline.

In [ ]:
def plot_divergence_from_baseline():
    candidates = [
        DATA_DIR / "datasets" / "swe_bench_verified_resolved_multifile" / "eval" / "divergence_results.json",
        DATA_DIR / "eval_results.json",
        DATA_DIR / "divergence_results.json",
        ROOT / "output" / "eval_results.json",
    ]
    div_path = next((p for p in candidates if p.exists()), None)
    if not div_path:
        print("Skipping: divergence_results.json not found. Run eval/run_eval.py --input output/resolved_traces_verified_multifile.jsonl --dataset swe_bench_verified_resolved_multifile")
        return
    with open(div_path) as f:
        div = json.load(f)
    per_proc = div.get("divergence_from_baseline", {}).get("per_procedure", {})
    if not per_proc:
        print("Skipping: no divergence_from_baseline in results.")
        return
    df = pd.DataFrame([{"repr": k, "mean_dist": v} for k, v in per_proc.items()])
    chart = (
        alt.Chart(df)
        .mark_bar(color="steelblue")
        .encode(alt.X("repr:N", title=""), alt.Y("mean_dist:Q", title="mean cosine dist from tokens"))
        .properties(title="Divergence from token baseline", width=280, height=200)
    )
    chart.save(PLOTS_DIR / "divergence_from_baseline.png")
    chart
plot_divergence_from_baseline()

## Procedural Divergence (Structural vs Semantic Gap)

Requires `procedure_divergence.parquet` (from `run_procedure_divergence.py`). Gap = structural_agreement − semantic_agreement. High gap = annotations add different info.

In [ ]:
def plot_procedure_divergence_gap():
    candidates = [
        DATA_DIR / "procedure_divergence.parquet",
        DATA_DIR / "datasets" / "swe_bench_lite" / "procedure_divergence.parquet",
    ]
    path = next((p for p in candidates if p.exists()), None)
    if not path:
        print("Skipping: procedure_divergence.parquet not found. Run run_procedure_divergence.py --output ...")
        return
    df = pd.read_parquet(path)
    df["pair"] = df["proc_a"] + " vs " + df["proc_b"]
    chart = (
        alt.Chart(df.dropna(subset=["gap"]))
        .mark_boxplot()
        .encode(
            alt.X("pair:N", title="procedure pair"),
            alt.Y("gap:Q", title="gap (structural − semantic agreement)"),
        )
        .properties(title="Procedural divergence gap by pair", width=350, height=220)
    )
    chart.save(PLOTS_DIR / "procedure_divergence_gap.png")
    chart
plot_procedure_divergence_gap()

## Retrieval Agreement (Instance-as-Query)

For each instance as query, rank others by distance under each representation. Spearman correlation between rankings = how much representations agree. Uses distance matrices from `build_distance_matrices.py`.

In [ ]:
def plot_retrieval_agreement():
    pair_path = DATA_DIR / "datasets" / "swe_bench_lite" / "per_instance_pair_rho.parquet"
    if not pair_path.exists():
        pair_path = DATA_DIR / "per_instance_pair_rho.parquet"
    if not pair_path.exists():
        print("Skipping: per_instance_pair_rho.parquet not found. Run run_diversity_analysis.py first.")
        return
    df = pd.read_parquet(pair_path)
    agg = df.groupby(["repr_i", "repr_j"])["rho"].mean().reset_index()
    agg["pair"] = agg["repr_i"] + " vs " + agg["repr_j"]
    chart = (
        alt.Chart(agg)
        .mark_bar(color="seagreen")
        .encode(alt.X("pair:N", sort="-y", title=""), alt.Y("rho:Q", title="mean Spearman ρ"))
        .properties(title="Retrieval agreement (instance-as-query)", width=320, height=220)
    )
    chart.save(PLOTS_DIR / "retrieval_agreement.png")
    chart
plot_retrieval_agreement()